# Universal Chat Templating with `apply_chat_template`

This notebook demonstrates `thinkpack.apply_chat_template` — a universal wrapper that produces correctly-formatted prompts for any model, regardless of how that model handles reasoning blocks internally.

Different thinking models inject the opening reasoning tag in different ways, and some actively strip reasoning from history turns:
- **Qwen3** (`prefixed=False`) — non-prefixed, `<think>` tags; the model generates the open tag itself.
- **Ministral** (`prefixed=False`, bracket tags) — non-prefixed, uses `[THINK]...[/THINK]` bracket tags instead of the common `<think>` style.
- **OLMo-3-Think** (`prefixed=True`) — the template appends `<think>` at the end of the generation prompt; the model generates the reasoning body itself.
- **DeepSeek-R1** (`prefixed=True`, `strips_think_tags=True`) — like OLMo-3, but the template also strips `<think>` blocks from assistant messages in multi-turn history. The library handles this via a sentinel approach so reasoning is always preserved.

Without a universal wrapper you would need different code for each model. With `apply_chat_template`, you write the same call once and it works correctly for all of them.

**What this notebook covers:**
1. `detect_model()` — auto-detecting template properties from a tokenizer
2. Basic inference prompts — passthrough, no reasoning tag enforced
3. Basic inference with reasoning — `add_generation_reasoning=True` ensures the open tag is always present
4. Simple instruction–response with reasoning — a completed single-turn exchange
5. Thought steering — seeding the think block at inference time
6. Response prefix — seeding the model's response
7. Combining both prefixes
8. Batching with `apply_chat_templates`

In [11]:
import sys

# add the src directory to the path so thinkpack can be imported without installation
sys.path.insert(0, "../src")

from transformers import AutoTokenizer

from thinkpack import apply_chat_template, apply_chat_templates, detect_model

## 1. Detecting Model Properties

`detect_model()` inspects a tokenizer's chat template to determine:
- **prefixed** — `True` if the template injects the opening reasoning tag into the generation prompt, `False` if the model generates it itself
- **open_tag / close_tag** — the reasoning block delimiters used by this model (`<think>` or `[THINK]` etc.)
- **strips_think_tags** — `True` if the template actively removes `<think>` blocks from assistant messages in history turns; the library compensates via a sentinel approach

Detection is entirely behaviour-based — no template source scanning or hardcoded model names.

In [12]:
# load tokenizers for all four models (cpu-only, no gpu needed)
tok_qwen = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3-8B",
    trust_remote_code=True,
)
tok_ministral = AutoTokenizer.from_pretrained(
    "mistralai/Ministral-3-8B-Reasoning-2512",
    trust_remote_code=True,
)
tok_olmo = AutoTokenizer.from_pretrained(
    "allenai/OLMo-3-7B-Think",
    trust_remote_code=True,
)
tok_deepseek = AutoTokenizer.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    trust_remote_code=True,
)

In [13]:
# detect template properties for each model — no manual configuration needed
for name, tok in [
    ("Qwen3", tok_qwen),
    ("Ministral", tok_ministral),
    ("OLMo-3", tok_olmo),
    ("DeepSeek-R1", tok_deepseek),
]:
    info = detect_model(tok)
    print(f"=== {name} ===")
    print(f"  prefixed         : {info.prefixed}")
    print(f"  open_tag         : {info.open_tag!r}")
    print(f"  close_tag        : {info.close_tag!r}")
    print(f"  strips_think_tags: {info.strips_think_tags}")
    print()

=== Qwen3 ===
  prefixed         : False
  open_tag         : '<think>'
  close_tag        : '</think>'
  strips_think_tags: False

=== Ministral ===
  prefixed         : False
  open_tag         : '[THINK]'
  close_tag        : '[/THINK]'
  strips_think_tags: False

=== OLMo-3 ===
  prefixed         : True
  open_tag         : '<think>'
  close_tag        : '</think>'
  strips_think_tags: False

=== DeepSeek-R1 ===
  prefixed         : True
  open_tag         : '<think>'
  close_tag        : '</think>'
  strips_think_tags: True



## 2. Basic Inference Prompts

The simplest use case: format a conversation ready for generation with no reasoning tag enforced. The same call works for all models — `apply_chat_template` returns the template output as-is by default (`add_generation_reasoning=None`).

For prefixed models (OLMo-3, DeepSeek-R1) the think tag is already at the end of the generation prompt. For non-prefixed models (Qwen3, Ministral) it is absent. Section 3 shows how to normalise this.

In [14]:
# a simple single-turn conversation
conversation = [
    {"role": "user", "content": "What is the capital of France?"},
]

# exact same call for all models
for name, tok in [
    ("Qwen3", tok_qwen),
    ("Ministral", tok_ministral),
    ("OLMo-3", tok_olmo),
    ("DeepSeek-R1", tok_deepseek),
]:
    prompt = apply_chat_template(
        conversation=conversation,
        tokenizer=tok,
    )
    print(f"=== {name} ===")
    print(prompt)
    print()

=== Qwen3 ===
<|im_start|>user
What is the capital of France?<|im_end|>


=== Ministral ===
<s>[SYSTEM_PROMPT]# HOW YOU SHOULD THINK AND ANSWER

First draft your thinking process (inner monologue) until you arrive at a response. Format your response using Markdown, and use LaTeX for any mathematical equations. Write both your thoughts and the response in the same language as the input.

Your thinking process must follow the template below:[THINK]Your thoughts or/and draft, like working through an exercise on scratch paper. Be as casual and as long as you want until you are confident to generate the response to the user.[/THINK]Here, provide a self-contained response.[/SYSTEM_PROMPT][INST]What is the capital of France?[/INST]

=== OLMo-3 ===
<|im_start|>system
You are OLMo, a helpful function-calling AI assistant built by Ai2. Your date cutoff is November 2024, and your model weights are available at https://huggingface.co/allenai. You do not currently have access to any functions. <fun

## 3. Basic Inference with Reasoning

Setting `add_generation_reasoning=True` ensures the opening think tag is present in the generation prompt for all models.

For **prefixed** models (OLMo-3, DeepSeek-R1) the tag is already injected by the template — `add_generation_reasoning=True` keeps it. For **non-prefixed** models (Qwen3, Ministral) the template does not add the tag — `add_generation_reasoning=True` appends it explicitly.

This is the most common inference mode for reasoning models: you always want the model to produce a think block before its answer.

In [15]:
# same conversation, but now with add_generation_reasoning=True — open think tag always present
conversation = [
    {"role": "user", "content": "What is the capital of France?"},
]

for name, tok in [
    ("Qwen3", tok_qwen),
    ("Ministral", tok_ministral),
    ("OLMo-3", tok_olmo),
    ("DeepSeek-R1", tok_deepseek),
]:
    prompt = apply_chat_template(
        conversation=conversation,
        tokenizer=tok,
        add_generation_reasoning=True,
    )
    print(f"=== {name} — tail of prompt ===")
    # show only the end where the think tag appears
    print(repr(prompt.split("assistant")[-1]))
    print()

=== Qwen3 — tail of prompt ===
'<|im_start|>user\nWhat is the capital of France?<|im_end|>\n<think>\n'

=== Ministral — tail of prompt ===
'<s>[SYSTEM_PROMPT]# HOW YOU SHOULD THINK AND ANSWER\n\nFirst draft your thinking process (inner monologue) until you arrive at a response. Format your response using Markdown, and use LaTeX for any mathematical equations. Write both your thoughts and the response in the same language as the input.\n\nYour thinking process must follow the template below:[THINK]Your thoughts or/and draft, like working through an exercise on scratch paper. Be as casual and as long as you want until you are confident to generate the response to the user.[/THINK]Here, provide a self-contained response.[/SYSTEM_PROMPT][INST]What is the capital of France?[/INST]\n[THINK]\n'

=== OLMo-3 — tail of prompt ===
' built by Ai2. Your date cutoff is November 2024, and your model weights are available at https://huggingface.co/allenai. You do not currently have access to any funct

## 4. Simple Instruction–Response with Reasoning

A completed single-turn exchange: user instruction followed by an assistant response that includes a `reasoning` key. This shows how the `reasoning` key embeds into the formatted sequence for a completed turn.

`add_generation_prompt=False` is passed so no trailing assistant opener is added — the exchange is already complete. This is the single-turn equivalent of the reasoning history shown in section 5.

In [16]:
# a completed single-turn exchange with reasoning in the assistant response
conversation = [
    {"role": "user", "content": "What is the capital of France?"},
    {
        "role": "assistant",
        "reasoning": "The capital of France is Paris — it has been the country's capital since the 10th century.",
        "content": "The capital of France is Paris.",
    },
]

for name, tok in [
    ("Qwen3", tok_qwen),
    ("Ministral", tok_ministral),
    ("OLMo-3", tok_olmo),
    ("DeepSeek-R1", tok_deepseek),
]:
    prompt = apply_chat_template(
        conversation=conversation,
        tokenizer=tok,
    )
    print(f"=== {name} ===")
    print(prompt)
    print()

=== Qwen3 ===
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
<think>
The capital of France is Paris — it has been the country's capital since the 10th century.
</think>

The capital of France is Paris.<|im_end|>


=== Ministral ===
<s>[SYSTEM_PROMPT]# HOW YOU SHOULD THINK AND ANSWER

First draft your thinking process (inner monologue) until you arrive at a response. Format your response using Markdown, and use LaTeX for any mathematical equations. Write both your thoughts and the response in the same language as the input.

Your thinking process must follow the template below:[THINK]Your thoughts or/and draft, like working through an exercise on scratch paper. Be as casual and as long as you want until you are confident to generate the response to the user.[/THINK]Here, provide a self-contained response.[/SYSTEM_PROMPT][INST]What is the capital of France?[/INST][THINK]
The capital of France is Paris — it has been the country's capital since the 10th cen

## 5. Thought Steering with `think_prefix`

At inference time, you may want to seed the model's reasoning — for example, to steer it towards a particular approach before it generates the rest of the think block itself.

`think_prefix` injects text at the start of the reasoning block. The model then continues generating from that point.

The same call works correctly for all four template styles:
- **`prefixed=False` (Qwen3, Ministral)** — the open tag is not yet in the prompt, so it is added before the prefix
- **`prefixed=True` (OLMo-3, DeepSeek-R1)** — the open tag is already at the end of the generation prompt, so only the body text is appended

In [17]:
# seed the model's thinking — the model will continue generating after this prefix
conversation = [
    {"role": "user", "content": "Is 97 a prime number?"},
]

for name, tok in [
    ("Qwen3", tok_qwen),
    ("Ministral", tok_ministral),
    ("OLMo-3", tok_olmo),
    ("DeepSeek-R1", tok_deepseek),
]:
    prompt = apply_chat_template(
        conversation=conversation,
        tokenizer=tok,
        think_prefix="Let me check by testing divisibility with primes up to √97 ≈ 9.8.",
    )
    print(f"=== {name} — tail of prompt ===")
    # show only the end of the prompt where the prefix was injected
    print(repr(prompt.split("assistant")[-1]))
    print()

=== Qwen3 — tail of prompt ===
'<|im_start|>user\nIs 97 a prime number?<|im_end|>\n<think>\nLet me check by testing divisibility with primes up to √97 ≈ 9.8.'

=== Ministral — tail of prompt ===
'<s>[SYSTEM_PROMPT]# HOW YOU SHOULD THINK AND ANSWER\n\nFirst draft your thinking process (inner monologue) until you arrive at a response. Format your response using Markdown, and use LaTeX for any mathematical equations. Write both your thoughts and the response in the same language as the input.\n\nYour thinking process must follow the template below:[THINK]Your thoughts or/and draft, like working through an exercise on scratch paper. Be as casual and as long as you want until you are confident to generate the response to the user.[/THINK]Here, provide a self-contained response.[/SYSTEM_PROMPT][INST]Is 97 a prime number?[/INST]\n[THINK]\nLet me check by testing divisibility with primes up to √97 ≈ 9.8.'

=== OLMo-3 — tail of prompt ===
' built by Ai2. Your date cutoff is November 2024, and y

## 6. Response Prefix Injection

`response_prefix` seeds the model's response after the reasoning block closes. Setting it implicitly closes the think block first, so the model is constrained to generate its response next.

This is useful for forcing a specific output format (e.g. JSON, a specific prefix like `"The answer is"`) or for constrained generation tasks.

In [18]:
# seed the response — think block closes automatically before the prefix
conversation = [
    {"role": "user", "content": "Is 97 a prime number?"},
]

for name, tok in [
    ("Qwen3", tok_qwen),
    ("Ministral", tok_ministral),
    ("OLMo-3", tok_olmo),
    ("DeepSeek-R1", tok_deepseek),
]:
    prompt = apply_chat_template(
        conversation=conversation,
        tokenizer=tok,
        response_prefix="The answer is: ",
    )
    print(f"=== {name} — tail of prompt ===")
    print(repr(prompt.split("assistant")[-1]))
    print()

=== Qwen3 — tail of prompt ===
'<|im_start|>user\nIs 97 a prime number?<|im_end|>\nThe answer is: '

=== Ministral — tail of prompt ===
'<s>[SYSTEM_PROMPT]# HOW YOU SHOULD THINK AND ANSWER\n\nFirst draft your thinking process (inner monologue) until you arrive at a response. Format your response using Markdown, and use LaTeX for any mathematical equations. Write both your thoughts and the response in the same language as the input.\n\nYour thinking process must follow the template below:[THINK]Your thoughts or/and draft, like working through an exercise on scratch paper. Be as casual and as long as you want until you are confident to generate the response to the user.[/THINK]Here, provide a self-contained response.[/SYSTEM_PROMPT][INST]Is 97 a prime number?[/INST]\nThe answer is: '

=== OLMo-3 — tail of prompt ===
' built by Ai2. Your date cutoff is November 2024, and your model weights are available at https://huggingface.co/allenai. You do not currently have access to any functions. 

## 7. Combining Thought Steering and Response Prefix

`think_prefix` and `response_prefix` can be used together. The model reasons from the seeded prefix, then the think block is closed and the response starts from the seeded prefix.

This is particularly useful in constrained evaluation settings — you can direct the model's reasoning strategy while also enforcing a structured response format.

In [19]:
# combine think and response prefixes — model reasons from the seed, then responds in format
conversation = [
    {"role": "user", "content": "Is 97 a prime number?"},
]

for name, tok in [
    ("Qwen3", tok_qwen),
    ("Ministral", tok_ministral),
    ("OLMo-3", tok_olmo),
    ("DeepSeek-R1", tok_deepseek),
]:
    prompt = apply_chat_template(
        conversation=conversation,
        tokenizer=tok,
        think_prefix="Let me check divisibility with primes up to √97 ≈ 9.8.",
        response_prefix="The answer is: ",
    )
    print(f"=== {name} — tail of prompt ===")
    print(repr(prompt.split("assistant")[-1]))
    print()

=== Qwen3 — tail of prompt ===
'<|im_start|>user\nIs 97 a prime number?<|im_end|>\n<think>\nLet me check divisibility with primes up to √97 ≈ 9.8.\n</think>\nThe answer is: '

=== Ministral — tail of prompt ===
'<s>[SYSTEM_PROMPT]# HOW YOU SHOULD THINK AND ANSWER\n\nFirst draft your thinking process (inner monologue) until you arrive at a response. Format your response using Markdown, and use LaTeX for any mathematical equations. Write both your thoughts and the response in the same language as the input.\n\nYour thinking process must follow the template below:[THINK]Your thoughts or/and draft, like working through an exercise on scratch paper. Be as casual and as long as you want until you are confident to generate the response to the user.[/THINK]Here, provide a self-contained response.[/SYSTEM_PROMPT][INST]Is 97 a prime number?[/INST]\n[THINK]\nLet me check divisibility with primes up to √97 ≈ 9.8.\n[/THINK]\nThe answer is: '

=== OLMo-3 — tail of prompt ===
' built by Ai2. Your dat

## 8. Batching with `apply_chat_templates`

`apply_chat_templates` (plural) accepts a list of conversations and processes them all in one call. `think_prefix` and `response_prefix` can be a single string (applied to every conversation) or a list with one entry per conversation — useful when you want different prefixes for different prompts.

In [20]:
# three questions processed in a single call with per-conversation think prefixes
conversations = [
    [{"role": "user", "content": "Is 97 a prime number?"}],
    [{"role": "user", "content": "What is 15% of 240?"}],
    [{"role": "user", "content": "What is the square root of 144?"}],
]

# different think seeds for each question
think_seeds = [
    "I'll test divisibility with small primes.",
    "I'll convert the percentage to a decimal first.",
    "I know that 12² = 144.",
]

# one call, three conversations, three different prefixes — works for any tokenizer
prompts = apply_chat_templates(
    conversations=conversations,
    tokenizer=tok_ministral,
    think_prefix=think_seeds,
)

for i, prompt in enumerate(prompts):
    print(f"=== Conversation {i + 1} — tail ===")
    print(repr(prompt.split("assistant")[-1]))
    print()

=== Conversation 1 — tail ===
"<s>[SYSTEM_PROMPT]# HOW YOU SHOULD THINK AND ANSWER\n\nFirst draft your thinking process (inner monologue) until you arrive at a response. Format your response using Markdown, and use LaTeX for any mathematical equations. Write both your thoughts and the response in the same language as the input.\n\nYour thinking process must follow the template below:[THINK]Your thoughts or/and draft, like working through an exercise on scratch paper. Be as casual and as long as you want until you are confident to generate the response to the user.[/THINK]Here, provide a self-contained response.[/SYSTEM_PROMPT][INST]Is 97 a prime number?[/INST]\n[THINK]\nI'll test divisibility with small primes."

=== Conversation 2 — tail ===
"<s>[SYSTEM_PROMPT]# HOW YOU SHOULD THINK AND ANSWER\n\nFirst draft your thinking process (inner monologue) until you arrive at a response. Format your response using Markdown, and use LaTeX for any mathematical equations. Write both your thoughts

## Summary

| Feature | Qwen3 (`prefixed=False`) | Ministral (`prefixed=False`, `[THINK]`) | OLMo-3 (`prefixed=True`) | DeepSeek-R1 (`prefixed=True`, `strips_think_tags`) |
|---|---|---|---|---|
| Basic prompt | Works | Works | Works | Works |
| `reasoning` history key | → inline `<think>` tags | → inline `[THINK]` tags | → inline `<think>` tags | → sentinel → re-injected `<think>` |
| `think_prefix` | Adds `<think>\n` then prefix | Adds `[THINK]\n` then prefix | Appends after existing `<think>` | Appends after existing `<think>` |
| `response_prefix` | Closes think block, adds prefix | Closes think block, adds prefix | Closes think block, adds prefix | Closes think block, adds prefix |
| Detection | Automatic | Automatic | Automatic | Automatic |

The key point: **you write the same code once**. `detect_model()` handles the per-model differences internally — including the sentinel workaround for `strips_think_tags` models — so your inference pipelines are identical regardless of which model family you are working with.